# POLAR OFFLINE RECORDER

## IMPORTS AND SETUPS

In [1]:
import yaml
from pathlib import Path
import platform
import asyncio
from bleak import BleakClient

In [2]:
# Config data are stored in config.yaml
configpath = Path("config.yaml")
with open (configpath, "r") as f:
        config = yaml.safe_load(f)

# BELT
'''
NOTE
Mac uses UUID, while Linux uses MAC address for the same task. So check the OS first.
Linux is called Linux. Windows is called Windows. Mac is called... wait for it... Darwin
'''
os = platform.system()
BELT = config["belt"]["uuid"] if os == "Darwin" else config["belt"]["mac_address"]
belt_human_readable = config["belt"]["name"]
# HEART RATE SERVICE (HRS)
HRS = config["belt"]["heart_rate_service"]

# POLAR MEASUREMENT DATA CONTROL (PMDC)
PMDC = config["belt"]["pmd_control"]

# POLAR MEASUREMENT DATA - DATA (PMDD)
PMDD = config["belt"]["pmd_data"]

## CONNECT

### Test available services

In [3]:
'''print("Connecting. This may take up to a minute.")

async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}.")

    print("Services:")
    for service in client.services:
        print(f"\nSERVICE {service.uuid}")
        for char in service.characteristics:
            print(f"  CHAR {char.uuid} | props={char.properties}")
    '''

'print("Connecting. This may take up to a minute.")\n\nasync with BleakClient(BELT) as client:\n    print(f"Connected to {belt_human_readable}.")\n\n    print("Services:")\n    for service in client.services:\n        print(f"\nSERVICE {service.uuid}")\n        for char in service.characteristics:\n            print(f"  CHAR {char.uuid} | props={char.properties}")\n    '

In [4]:
ACC_MEASUREMENT_TYPE = 0x02
ECG_MEASUREMENT_TYPE = 0x00

ACC_GET_SETTINGS = bytearray([0x01, ACC_MEASUREMENT_TYPE])
ECG_GET_SETTINGS = bytearray([0x01, ECG_MEASUREMENT_TYPE])


In [5]:
def handle_pmd_control(sender, data):
    print("PMD CONTROL:", data.hex(" "))

In [6]:
'''async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}")

    try:
        await asyncio.wait_for(
            client.start_notify(PMDC, handle_pmd_control),
            timeout=5
        )
        print("PMDC indication subscription started")

    except asyncio.TimeoutError:
        print("TIMEOUT: start_notify on PMDC did not finish")'''

'async with BleakClient(BELT) as client:\n    print(f"Connected to {belt_human_readable}")\n\n    try:\n        await asyncio.wait_for(\n            client.start_notify(PMDC, handle_pmd_control),\n            timeout=5\n        )\n        print("PMDC indication subscription started")\n\n    except asyncio.TimeoutError:\n        print("TIMEOUT: start_notify on PMDC did not finish")'

In [7]:
BATTERY_LEVEL = "00002a19-0000-1000-8000-00805f9b34fb"

async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}")

    battery = await client.read_gatt_char(BATTERY_LEVEL)
    print("Battery:", battery[0], "%")

BleakError: failed to discover services, device disconnected